# PyTorch Quest

*From rusty Keras to reading a DQN robot project and research papers*

**How this works.** 16 short levels. Each one has: a hook, a tiny idea, code you run in Colab, and a line saying where it shows up in your DQN TurtleBot project. Expected outputs are written in comments, so you can guess first and then check.

**Two files go together:** this PDF is for reading, `PyTorch_Quest.ipynb` is the same course as a Colab notebook (upload it at colab.research.google.com, File > Upload notebook). Run the cells top to bottom; later cells reuse earlier variables.

**Rule of thumb:** do not read a level without running its code. 10 minutes per level is plenty.

**The map**
- Levels 0-3: tensors, shapes, gradients (the raw material)
- Levels 4-7: models, loss, training loop, data (the engine)
- Levels 8-9: GPU, saving and loading (the plumbing that breaks)
- Levels 10-12: CNNs, research-code patterns, a bug-hunting game
- Levels 13-14: DQN bridge to your project, reading papers and equations
- Level 15: installing on your VirtualBox Ubuntu. Level 16: cheat sheet and roadmap

## Level 0: Is the machine awake?

> **Hook:** Before any deep learning: does PyTorch see a GPU, and which one?

In Colab: Runtime > Change runtime type > pick a GPU (T4). Then run:

In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())            # True on a Colab GPU runtime
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))    # e.g. Tesla T4

device = "cuda" if torch.cuda.is_available() else "cpu"   # you will see this line in every repo
print("using:", device)

2.11.0+cu130
True
Tesla T4
using: cuda


> **In your project:** Your VirtualBox VM will print False here (VirtualBox cannot hand your GPU to the VM). That is fine for this project: its network is small enough to run on CPU.

## Level 1: Tensors, the Keras arrays with superpowers

> **Hook:** Why does torch.tensor([1,2,3]) / 2 not give integers? And what does x[:, 1] even mean?

In [2]:
import torch
a = torch.tensor([1, 2, 3])            # ints   -> int64
b = torch.tensor([1.0, 2.0, 3.0])      # floats -> float32  (the default for models)
print(a.dtype, b.dtype)                # torch.int64 torch.float32

z = torch.zeros(2, 3); o = torch.ones(2, 3); r = torch.rand(2, 3); n = torch.randn(2, 3)
print(z.shape, z.dtype)                # torch.Size([2, 3]) torch.float32

x = torch.arange(12).reshape(3, 4)
print(x)                               # [[0,1,2,3],[4,5,6,7],[8,9,10,11]]
print(x[0])                            # tensor([0, 1, 2, 3])          first row
print(x[:, 1])                         # tensor([1, 5, 9])             second column
print(x[1:, :2])                       # tensor([[4, 5],[8, 9]])       rows 1.., cols ..2
print(x[x > 5])                        # tensor([ 6,  7,  8,  9, 10, 11])   boolean mask
print(x[:, -1].shape)                  # torch.Size([3])
print(x[None].shape)                   # torch.Size([1, 3, 4])   None adds a new axis

print(torch.tensor([1, 2, 3]) / 2)     # tensor([0.5000, 1.0000, 1.5000])  / always gives floats

torch.int64 torch.float32
torch.Size([2, 3]) torch.float32
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([0, 1, 2, 3])
tensor([1, 5, 9])
tensor([[4, 5],
        [8, 9]])
tensor([ 6,  7,  8,  9, 10, 11])
torch.Size([3])
torch.Size([1, 3, 4])
tensor([0.5000, 1.0000, 1.5000])


> **AHA:** A tensor has three identity cards: **shape**, **dtype**, **device**. Nine out of ten PyTorch errors are one of these three not matching.

> **Keras:** Same idea as a TensorFlow tensor or NumPy array. `x.numpy()` and `torch.from_numpy(arr)` convert (they can share memory on CPU).

**Practice** (try before reading the answer in the comment)

In [3]:
t = torch.arange(24).reshape(2, 3, 4)
print(t.sum(dim=-1).shape)             # torch.Size([2, 3])   the last axis disappears
print(t[:, 0, :].shape)                # torch.Size([2, 4])
print(t.float().mean().item())         # 11.5   .item() turns a 1-element tensor into a Python number

torch.Size([2, 3])
torch.Size([2, 4])
11.5


> **In your project:** In the DQN project, a robot state (lidar + goal info) is a 1-D float tensor, the chosen action is an integer, and rewards are floats. Mixing those dtypes up is the classic first bug.

## Level 2: Shape surgery (the #1 skill)

> **Hook:** Predict: a has shape [5], b has shape [5, 1]. What is the shape of a - b?

In [4]:
x = torch.arange(6)
print(x)                                  # [6]
print(x.view(2, 3).shape)                            # [2, 3]
print(x.reshape(3, -1).shape)                        # [3, 2]    -1 means "figure it out"

v = torch.tensor([1., 2., 3.])
print(v.unsqueeze(0).shape, v.unsqueeze(1).shape)    # [1, 3] [3, 1]
print(v[:, None].shape)
print(v[None,:].shape)                              # [3, 1]    same as unsqueeze(1)
print(v.unsqueeze(1).squeeze(1).shape)               # [3]       squeeze removes size-1 axes

a = torch.ones(2, 3); b = torch.zeros(2, 3)
print(torch.cat([a, b], dim=0).shape)                # [4, 3]    join along an existing axis
print(torch.cat([a, b], dim=1).shape)                # [2, 6]
print(torch.stack([a, b]).shape)                     # [2, 2, 3] creates a NEW axis

tensor([0, 1, 2, 3, 4, 5])
torch.Size([2, 3])
torch.Size([3, 2])
torch.Size([1, 3]) torch.Size([3, 1])
torch.Size([3, 1])
torch.Size([1, 3])
torch.Size([3])
torch.Size([4, 3])
torch.Size([2, 6])
torch.Size([2, 2, 3])


**Broadcasting**: line shapes up from the right; each pair of sizes must be equal or one of them must be 1 (or missing). The size-1 side gets stretched.

In [5]:
m   = torch.ones(4, 3)
row = torch.tensor([1., 2., 3.])                     # [3]
col = torch.tensor([[10.], [20.], [30.], [40.]])     # [4, 1]
print((m + row).shape)                               # [4, 3]
print((m + col).shape)                               # [4, 3]
print((row + col).shape)                             # [4, 3]  [3] with [4,1] -> grid

a = torch.randn(5); b = torch.randn(5, 1)
print((a - b).shape)                                 # [5, 5] !!! the answer to the hook

torch.Size([4, 3])
torch.Size([4, 3])
torch.Size([4, 3])
torch.Size([5, 5])


> **TRAP:** The [5] minus [5,1] surprise silently makes a 5x5 grid. It is the most common silent bug in loss code (predictions [B] vs targets [B,1]). Habit: **print shapes before and after any subtraction**.

In [6]:
x = torch.randn(32, 10); w = torch.randn(10, 4)
print((x @ w).shape)                                 # [32, 4]   matmul: [B,N] @ [N,M]
q = torch.randn(4, 3)                               # 32 states, 5 action scores each
print(q)
print(q.max(dim=1).values.shape)                     # [32]      best score per row
print(q.argmax(dim=1).shape)                         # [32]      best action per row
print(q.sum(dim=1, keepdim=True).shape)              # [32, 1]   keepdim keeps the axis

actions = torch.randint(0, 2, (4,))                 # one chosen action per row
print(actions)
chosen = q.gather(1, actions.unsqueeze(1)).squeeze(1)
print(chosen.shape)                                  # [32]  the Q-value of the action actually taken
print(chosen)

torch.Size([32, 4])
tensor([[ 0.0652, -1.5141, -2.4019],
        [ 0.0429,  2.2397, -1.3603],
        [ 0.0419,  1.0776,  0.7557],
        [-0.3845, -0.7475,  2.6397]])
torch.Size([4])
torch.Size([4])
torch.Size([4, 1])
tensor([1, 1, 0, 1])
torch.Size([4])
tensor([-1.5141,  2.2397,  0.0419, -0.7475])


> **In your project:** `gather` is how DQN picks the Q-value of the action that was taken. You will see this exact line in the training code.

## Level 3: Autograd, gradients for free

> **Hook:** We never write a derivative. So how does PyTorch know the gradient?

In [7]:
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()
print(x.grad)              # tensor(6.)   because dy/dx = 2x = 6

w = torch.tensor([2.0, -1.0], requires_grad=True)
x = torch.tensor([3.0, 4.0])
y = (w * x).sum()          # 2*3 + (-1)*4 = 2
y.backward()
print(w.grad)              # tensor([3., 4.])   dy/dw = x

tensor(6.)
tensor([3., 4.])


> **AHA:** PyTorch silently records every operation on tensors that require gradients (a graph). `backward()` walks the graph backwards and **adds** the results into `.grad`.

In [8]:
w = torch.tensor(1.0, requires_grad=True)
for _ in range(3):
    (w * 2).backward()
print(w.grad)              # tensor(6.)  = 2 + 2 + 2, gradients ACCUMULATE
w.grad.zero_()             # this is why training loops call optimizer.zero_grad()

with torch.no_grad():
    y = w * 2
print(y.requires_grad)     # False: no graph recorded (use for inference, targets)

y2 = (w * 2).detach()      # keeps the value, cuts the graph
print(y2.requires_grad)    # False

tensor(6.)
False
False


> **Keras:** `tf.GradientTape` = autograd, but always on. No tape block: you just call `loss.backward()`.

> **In your project:** In DQN the target value `r + gamma * max Q_target(s')` is computed inside `torch.no_grad()` so only the online network learns. Look for `no_grad` and `detach` in the repo.

## Level 4: Learning by hand (no nn module yet)

> **Hook:** Training is just: guess, measure the error, nudge the numbers downhill, repeat. Let us do exactly that.

In [ ]:
torch.manual_seed(0)
x = torch.linspace(-1, 1, 50).unsqueeze(1)           # [50, 1]
y = 3 * x + 1 + 0.1 * torch.randn_like(x)            # secret rule: y = 3x + 1, plus noise

w = torch.zeros(1, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.1

for step in range(200):
    pred = x @ w + b                                  # [50, 1]
    loss = ((pred - y) ** 2).mean()                   # mean squared error
    loss.backward()                                   # fills w.grad and b.grad
    with torch.no_grad():
        w -= lr * w.grad                              # the gradient descent update
        b -= lr * b.grad
    w.grad.zero_(); b.grad.zero_()

print(w.item(), b.item(), loss.item())                # about 3.0, 1.0, 0.01

> **AHA:** Everything later (nn.Module, optimizers, DQN) is this loop with nicer wrappers: forward, loss, backward, update, zero grads.

## Level 5: nn.Module, reading a model class

> **Hook:** Every research repo is mostly classes that inherit nn.Module. Learn to read one top to bottom.

In [ ]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        self.fc1 = nn.Linear(n_in, n_hidden)       # layers are created (and registered) here
        self.fc2 = nn.Linear(n_hidden, n_out)
    def forward(self, x):                          # x: [B, n_in]   how data flows
        x = torch.relu(self.fc1(x))                # [B, n_hidden]
        return self.fc2(x)                         # [B, n_out]

model = MLP(4, 8, 3)
print(model)
for name, p in model.named_parameters():
    print(name, tuple(p.shape))
# fc1.weight (8, 4)   fc1.bias (8,)   fc2.weight (3, 8)   fc2.bias (3,)
print(sum(p.numel() for p in model.parameters()))  # 67 = (4*8+8) + (8*3+3)
print(model(torch.randn(5, 4)).shape)              # [5, 3]   call model(x), never model.forward(x)

> **AHA:** Weights are stored as **(out, in)**, the opposite of Keras kernels (in, out). Do not panic when shapes look transposed.

In [ ]:
class Bad(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(2, 2)]            # plain Python list
print(len(list(Bad().parameters())))               # 0  !!! the optimizer would see nothing

class Good(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(2, 2)])
print(len(list(Good().parameters())))              # 2  (weight and bias)

seq = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 3))   # shortcut for plain stacks
print(seq(torch.randn(5, 4)).shape)                # [5, 3]

> **Keras:** `nn.Module` plays the role of both `Layer` and `Model`. `nn.Sequential` is `keras.Sequential`. There is no `compile()` or `fit()`: you write the loop (next level).

> **In your project:** The DQN network is a small stack of Linear layers: state in, one score per action out, with no softmax at the end (Q-values are not probabilities).

## Level 6: The training loop (the heartbeat)

> **Hook:** model.fit() hid five lines. Here they are.

In [ ]:
import torch, torch.nn as nn
torch.manual_seed(0)
n = 200
X = torch.cat([torch.randn(n, 2) + 2, torch.randn(n, 2) - 2])      # two blobs in 2-D
y = torch.cat([torch.zeros(n), torch.ones(n)]).long()               # class labels (must be long)
perm = torch.randperm(2 * n); X, y = X[perm], y[perm]
Xtr, ytr, Xva, yva = X[:300], y[:300], X[300:], y[300:]

model = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
loss_fn = nn.CrossEntropyLoss()                                     # takes raw scores [B,C] + labels [B]
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(20):
    model.train()
    logits = model(Xtr)                    # 1. forward
    loss = loss_fn(logits, ytr)            # 2. loss
    opt.zero_grad()                        # 3. clear old gradients
    loss.backward()                        # 4. gradients
    opt.step()                             # 5. update weights

    model.eval()
    with torch.no_grad():                  # validation: no graph
        acc = (model(Xva).argmax(1) == yva).float().mean().item()
    if epoch % 5 == 0:
        print(epoch, round(loss.item(), 3), acc)                    # loss falls, accuracy near 1.0

> **AHA:** Memorize the heartbeat: **forward, loss, zero_grad, backward, step**. Every training script in every paper repo is this, decorated.

- `model.train()` / `model.eval()` flip behaviour of dropout and batch-norm. Use them anyway.
- `torch.no_grad()` for any evaluation or action selection.
- Common optimizers: `SGD`, `Adam`, `AdamW`. Learning-rate schedulers (`torch.optim.lr_scheduler`) just change `lr` over time.

> **In your project:** DQN has the same five lines. Differences: the loss compares Q(s,a) to a bootstrapped target instead of a label, and the data comes from a replay buffer instead of a dataset.

## Level 7: Dataset and DataLoader (batching on autopilot)

> **Hook:** Real data does not fit in one tensor. How does a repo feed it to the model?

In [ ]:
from torch.utils.data import Dataset, DataLoader

class Blobs(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):                     # how many samples
        return len(self.X)
    def __getitem__(self, i):              # sample number i
        return self.X[i], self.y[i]

loader = DataLoader(Blobs(Xtr, ytr), batch_size=32, shuffle=True)
xb, yb = next(iter(loader))
print(xb.shape, yb.shape)                  # [32, 2] [32]
print(len(loader))                         # 10 batches (the last one has only 12 samples)

for epoch in range(3):
    for xb, yb in loader:                  # the usual double loop: epochs, then batches
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()

Trace in any repo: **raw data > preprocessing > Dataset > DataLoader > batch > model**. Useful DataLoader options: `num_workers` (parallel loading), `collate_fn` (how samples are glued into a batch, needed for variable-length data).

> **In your project:** Reinforcement learning has no fixed dataset. The **replay buffer** takes the DataLoader's job: it stores past (state, action, reward, next_state, done) and hands back random batches.

## Level 8: GPUs, devices and memory

> **Hook:** Your laptop has 6 GB of VRAM. What eats it, and what are the classic crashes?

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)                   # moves weights (in place for modules)
xb = xb.to(device)                         # tensors are NOT in place: you must reassign!
print(next(model.parameters()).device, xb.device)

# The classic crash (only demonstrates on a GPU runtime):
if device == "cuda":
    try:
        model(torch.randn(4, 2))           # input still on CPU
    except RuntimeError as e:
        print("Caught:", str(e)[:70])      # "Expected all tensors to be on the same device..."

if device == "cuda":
    print(torch.cuda.memory_allocated() / 1e6, "MB in use")
    print(torch.cuda.max_memory_allocated() / 1e6, "MB peak")

**What fills VRAM:** model weights, activations saved for backward (grows with batch size and image size), gradients, optimizer state (Adam keeps 2 extra copies of every weight).

**If you hit `CUDA out of memory`:** lower the batch size; evaluate under `torch.no_grad()`; use mixed precision (`torch.autocast`); accumulate gradients over several small batches; `del` big tensors you no longer need.

> **In your project:** The DQN network is tiny (about a megabyte of weights) so 6 GB is not a problem here; memory only becomes a concern with large CNN models or huge replay buffers kept on the GPU.

## Level 9: Saving and loading (where real projects break)

> **Hook:** The README says: download the pretrained model. What is inside that file, and why does loading fail so often?

In [ ]:
torch.save(model.state_dict(), "model.pt")
print(list(model.state_dict().keys()))     # ['0.weight', '0.bias', '2.weight', '2.bias']

new = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))   # same architecture first!
new.load_state_dict(torch.load("model.pt", map_location="cpu"))
new.eval()                                  # inference mode

# A fuller checkpoint, to resume training later
ckpt = {"model": model.state_dict(), "opt": opt.state_dict(), "epoch": 20}
torch.save(ckpt, "ckpt.pt")

# What a mismatch looks like:
wrong = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 2))
try:
    wrong.load_state_dict(torch.load("model.pt", map_location="cpu"))
except RuntimeError as e:
    print("size mismatch caught:", "size mismatch" in str(e))        # True

- A **state_dict** is a dict of layer name to weight tensor. The code must build the same architecture before loading.
- `map_location="cpu"` lets a GPU-saved file load on a CPU-only machine (likely relevant in your VM).
- `strict=False` ignores missing or extra keys. Handy for partial loading, dangerous because it hides mistakes.
- PyTorch 2.6 and newer load with `weights_only=True` by default; old files holding whole pickled objects may need `weights_only=False` (only for files you trust). The project's README pins PyTorch 2.2.2.

> **In your project:** `ros2 run enpm690_dqn dqn_test dqn_2_stage_4 2600` most likely loads a saved model by name and episode number [verify in the code]. If you see size mismatch errors, the state size or layer sizes in the code differ from those used when the file was saved.

## Level 10: CNN quickie (papers love images)

> **Hook:** Keras puts channels last. PyTorch puts them first. Predict the output shape of a conv layer.

In [ ]:
conv = nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, stride=1, padding=1)
x = torch.randn(4, 3, 32, 32)              # [B, C, H, W]  channels FIRST
print(conv(x).shape)                       # [4, 8, 32, 32]
pool = nn.MaxPool2d(2)
h = pool(conv(x))
print(h.shape)                             # [4, 8, 16, 16]
print(h.flatten(1).shape)                  # [4, 2048] = 8*16*16, ready for nn.Linear

Output size of a conv: `floor((H + 2*padding - kernel) / stride) + 1`. Check: (32 + 2 - 3)/1 + 1 = 32.

> **In your project:** This project feeds lidar numbers rather than images, so it uses plain Linear layers. But papers on visual navigation use CNN encoders, and you will meet this shape pattern there.

## Level 11: Patterns you will meet in research code

> **Hook:** The code looks scary because of a handful of repeated tricks. Meet them.

In [ ]:
# 1. Residual connection: output = input + something
class ResBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.f = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, d))
    def forward(self, x):
        return x + self.f(x)
print(ResBlock(16)(torch.randn(5, 16)).shape)          # [5, 16]  shape must not change

# 2. Forward hooks: print every layer's shapes without editing the model
probe = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))
def show(m, inp, out):
    print(m.__class__.__name__, tuple(inp[0].shape), "->", tuple(out.shape))
for layer in probe:
    layer.register_forward_hook(show)
probe(torch.randn(5, 2))
# Linear (5, 2) -> (5, 16) / ReLU (5, 16) -> (5, 16) / Linear (5, 16) -> (5, 2)

# 3. register_buffer: saved and moved to GPU with the model, but not trained
class Scaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("scale", torch.tensor(10.0))
    def forward(self, x):
        return x / self.scale
print(list(Scaler().state_dict().keys()))              # ['scale']  buffers appear in state_dict

# 4. einsum: shape arithmetic in one line (b=batch, i=in, j=out)
x = torch.randn(32, 10); w = torch.randn(10, 4)
print(torch.allclose(torch.einsum("bi,ij->bj", x, w), x @ w))   # True

# 5. masks: write -inf (or a big negative) where something is not allowed
s = torch.tensor([1.0, 2.0, 3.0]); mask = torch.tensor([True, False, True])
print(s.masked_fill(~mask, -1e9).argmax())             # tensor(2)

Also common: `@torch.no_grad()` as a decorator, `argparse` / YAML config files, `torch.manual_seed(...)` for reproducibility, `if __name__ == '__main__':` entry points, `nn.Module` classes nested inside other `nn.Module` classes.

## Level 12: Bug-hunting game

> **Hook:** Each cell is broken on purpose. Guess the bug from the error, then read the verdict under it.

In [ ]:
# Bug 1
try:
    nn.Linear(10, 5)(torch.randn(3, 8))
except RuntimeError as e:
    print("Bug 1:", e)
# Verdict: input has 8 features but the layer expects 10. Print x.shape and the first layer.

# Bug 2
try:
    nn.CrossEntropyLoss()(torch.randn(2, 3), torch.tensor([0., 1.]))
except Exception as e:
    print("Bug 2:", type(e).__name__)
# Verdict: class labels must be integer (long) tensors, not floats. Use .long().

# Bug 3  (silent!)
w = torch.tensor(1.0, requires_grad=True)
for i in range(3):
    (w * 2).backward()                 # no zero_grad
    print("Bug 3 grad:", w.grad.item())          # 2.0, 4.0, 6.0 -- should stay 2.0
# Verdict: gradients accumulate. Call opt.zero_grad() every step.

# Bug 4
try:
    (w * 2).detach().sum().backward()
except RuntimeError as e:
    print("Bug 4:", str(e)[:60])
# Verdict: you cut the graph with detach(); nothing left to differentiate.

# Bug 5  (silent!)
drop = nn.Sequential(nn.Linear(4, 4), nn.Dropout(0.5))
x = torch.ones(1, 4)
drop.train(); print("train:", drop(x))        # random zeros, changes every run
drop.eval();  print("eval: ", drop(x), drop(x))   # identical both times
# Verdict: forgetting model.eval() at test time gives noisy, worse results.

# Bug 6
a = torch.randn(5); b = torch.randn(5, 1)
print("Bug 6 shape:", (a - b).shape)           # [5, 5]: silent broadcasting. Use b.squeeze(1).

**Debug recipe:** (1) read the last line of the error, (2) print `.shape`, `.dtype`, `.device` of every input, (3) test with batch size 2 and a tiny model, (4) check for NaN with `torch.isnan(x).any()`, (5) for "model does not learn": overfit one tiny batch first. If it cannot, the bug is in the code, not the data.

## Level 13: Bridge to your project, a complete mini DQN

> **Hook:** A robot on a line of 6 cells must walk from cell 0 to cell 5. Same ingredients as the TurtleBot project, no ROS needed.

The only new ideas (details in the companion PDF):
- **State** s (numbers the agent sees), **action** a (integer), **reward** r (float).
- **Q-network**: state in, one score per action out. Greedy policy: `argmax`.
- **Target** for a transition: `y = r + gamma * max_a' Q_target(s', a') * (1 - done)`. Loss: Huber between `Q(s,a)` and `y`.
- **Replay buffer** (random batches), **target network** (slow copy), **epsilon-greedy** (random exploration that decays).

In [ ]:
import random, collections
device = "cuda" if torch.cuda.is_available() else "cpu"

class LineEnv:
    def __init__(self, n=6): self.n = n
    def reset(self):
        self.pos = 0
        return self._state()
    def _state(self):                                   # shape [1]
        return torch.tensor([self.pos / (self.n - 1)], dtype=torch.float32)
    def step(self, a):                                  # 0 = left, 1 = right
        self.pos = max(0, min(self.n - 1, self.pos + (1 if a == 1 else -1)))
        done = self.pos == self.n - 1
        return self._state(), (1.0 if done else -0.01), done

class QNet(nn.Module):
    def __init__(self, s=1, a=2, h=32):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(s, h), nn.ReLU(), nn.Linear(h, h), nn.ReLU(), nn.Linear(h, a))
    def forward(self, x): return self.net(x)

q, q_tgt = QNet().to(device), QNet().to(device)
q_tgt.load_state_dict(q.state_dict())
opt = torch.optim.Adam(q.parameters(), lr=1e-3)
buffer = collections.deque(maxlen=10_000)
gamma, eps, B = 0.9, 1.0, 64
env = LineEnv()

for ep in range(400):
    s = env.reset()
    for t in range(60):
        if random.random() < eps:
            a = random.randrange(2)                           # explore
        else:
            with torch.no_grad():
                a = q(s.to(device)).argmax().item()           # exploit
        s2, r, done = env.step(a)
        buffer.append((s, a, r, s2, float(done)))
        s = s2
        if len(buffer) >= B:
            S, A, R, S2, D = zip(*random.sample(buffer, B))
            S  = torch.stack(S).to(device)                    # [B, 1]
            A  = torch.tensor(A, device=device)               # [B]   long
            R  = torch.tensor(R, dtype=torch.float32, device=device)
            S2 = torch.stack(S2).to(device)
            D  = torch.tensor(D, dtype=torch.float32, device=device)
            q_sa = q(S).gather(1, A.unsqueeze(1)).squeeze(1)  # [B]
            with torch.no_grad():
                y = R + gamma * q_tgt(S2).max(1).values * (1 - D)
            loss = nn.functional.smooth_l1_loss(q_sa, y)
            opt.zero_grad(); loss.backward(); opt.step()
        if done: break
    eps = max(0.05, eps * 0.98)
    if ep % 10 == 0:
        q_tgt.load_state_dict(q.state_dict())

s, path = env.reset(), []
for _ in range(20):
    with torch.no_grad():
        a = q(s.to(device)).argmax().item()
    s, r, done = env.step(a); path.append(a)
    if done: break
print(path)                                  # hoped: [1, 1, 1, 1, 1] (random: rerun if needed)

> **In your project:** Map it: `LineEnv` ~ the environment node (state, reward, done); `QNet` ~ the model class; the loop ~ the training script; the last cell ~ `dqn_test`. When you open the repo, find each of these four pieces.

## Level 14: Reading papers and equations without fear

> **Hook:** An equation is just code written by someone in a hurry. Translate it.

**Three-pass reading**
1. **5 minutes:** abstract, figures, conclusion. What problem, what trick, what result?
2. **30 minutes:** method section. Draw the architecture. For every equation list: each symbol, its shape, what it means in words.
3. **Only if needed:** details, proofs, appendix, the code.

**Equation to code template** (fill in for each important equation):
`symbol > shape > meaning > file:function > PyTorch line`

**Math you actually need, verified in code**

In [ ]:
import torch.nn.functional as F
logits = torch.tensor([2.0, 1.0, 0.1])
p = torch.softmax(logits, dim=0)
print(p)                                   # tensor([0.6590, 0.2424, 0.0986])  sums to 1
ce = F.cross_entropy(logits[None], torch.tensor([0]))      # true class = 0
print(ce.item(), -torch.log(p[0]).item())  # both about 0.417: cross-entropy = -log(prob of true class)

# Gradient descent update:   w <- w - lr * dLoss/dw    (Level 4)
# MSE loss:                  mean((pred - target)^2)
# Bellman target (DQN):      y = r + gamma * max_a' Q_target(s', a')   (Level 13)

**Keep a paper-to-code map** for each paper you study:

| Paper idea | Equation | File | Function/class |
|---|---|---|---|
| Q-network | | | |
| Replay buffer | | | |
| Target network | | | |
| Reward | | | |

**Reading list for this project:** the DQN paper (Mnih et al., Nature 2015); the upstream `tomasvr/turtlebot3_drlnav` repo and its associated research; the group's report PDF in your repo. Then branch out: Double DQN, Dueling DQN, prioritized replay, DDPG/TD3 (continuous control).

## Level 15: Installing PyTorch on your VirtualBox Ubuntu

> **Hook:** Colab first. Come here only when you want to run the project locally.

**Reality check.** VirtualBox does not pass your RTX 3050 into the VM, so PyTorch inside the VM runs on CPU (`torch.cuda.is_available()` is `False`). That is OK: the project's network is small. Use Colab (or native Linux) for anything heavy.

**VM settings that matter:** Ubuntu **22.04** (ROS 2 Humble requires it, with Python 3.10); at least 4 GB RAM (more if your host allows), 2-4 CPU cores, 40+ GB disk; install Guest Additions. Gazebo may be slow in a VM (software rendering); that is normal.

**Step 1: base tools**

```bash
sudo apt update
sudo apt install -y python3-pip python3-venv git
python3 --version        # should say 3.10.x
```

**Step 2a: for learning (isolated environment)**

```bash
python3 -m venv ~/torch-env
source ~/torch-env/bin/activate
pip install --upgrade pip
pip install "numpy<2" matplotlib pandas jupyter
pip install torch==2.2.2 --index-url https://download.pytorch.org/whl/cpu
python -c "import torch; print(torch.__version__, torch.cuda.is_available())"
# expected: 2.2.2+cpu False
```

**Step 2b: for running the ROS 2 project.** ROS nodes started with `ros2 run` use the system Python, which does not see packages inside a venv. Simplest fix: install the same versions for the system Python (deactivate any venv first):

```bash
deactivate 2>/dev/null
pip3 install --user "numpy<2" matplotlib pandas
pip3 install --user torch==2.2.2 --index-url https://download.pytorch.org/whl/cpu
python3 -c "import torch; print(torch.__version__)"
```

Why `numpy<2`: PyTorch 2.2.2 and many ROS 2 Humble packages were built against NumPy 1.x, and NumPy 2 produces confusing import errors.

**Then follow the project README** (ROS 2 Humble, Gazebo, `colcon build`, the `.bashrc` lines). If the code insists on `cuda`, change the device line to `"cuda" if torch.cuda.is_available() else "cpu"` and load checkpoints with `map_location="cpu"`.

**Common errors**
- `ModuleNotFoundError: torch` when running a node: torch is in the wrong Python. Use Step 2b.
- `A module compiled using NumPy 1.x cannot be run in NumPy 2`: run `pip3 install --user "numpy<2"`.
- `RuntimeError: ... deserialize object on a CUDA device`: add `map_location="cpu"` to `torch.load`.
- Gazebo black screen or crashes: lower VM resolution, give more RAM/video memory, or run the sim on a native Ubuntu install.

**If you later want the 3050 for real (native Ubuntu, dual boot, not a VM):**

```bash
sudo ubuntu-drivers autoinstall       # installs the NVIDIA driver, then reboot
nvidia-smi                            # should list the RTX 3050 and the driver version
pip install torch==2.2.2 --index-url https://download.pytorch.org/whl/cu121
python -c "import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))"
```

The CUDA build of PyTorch ships its own CUDA runtime, so you only need the NVIDIA driver, not a separate CUDA toolkit. With 6 GB of VRAM: keep batches small, use `torch.no_grad()` for evaluation, and try mixed precision.

## Level 16: Cheat sheet and what to do next

**The heartbeat**
```
for batch in loader:
    pred = model(x)            # forward
    loss = loss_fn(pred, y)    # loss
    opt.zero_grad()            # clear
    loss.backward()            # gradients
    opt.step()                 # update
```

**Three identity cards of a tensor:** shape, dtype, device.

**Shape reflexes:** Linear works on the last axis. `unsqueeze` adds an axis, `squeeze` removes size-1 axes. `cat` joins on an existing axis, `stack` makes a new one. Broadcasting aligns from the right.

**Keras to PyTorch**

| Keras / TF | PyTorch |
|---|---|
| `Model`, `Layer` | `nn.Module` |
| `model.fit` | your loop |
| `GradientTape` | `loss.backward()` |
| `tf.data` | `Dataset` + `DataLoader` |
| `save_weights` | `state_dict` + `torch.save` |
| channels last | channels first |

**Your next steps, in order**
1. Run Levels 0-9 in Colab (about 90 minutes total).
2. Run Level 13 until it solves the line world; then break it on purpose (remove the target network, shrink the buffer, change gamma).
3. Unzip the project; find the model class, the state vector, the reward, the save/load code (use the checklist in the companion PDF).
4. Paste the model class and reward function to me and I will annotate them line by line.
5. Read the DQN paper with the Level 14 method; fill in the paper-to-code table.
6. Then explore: Double DQN (a one-line change to the target), a different reward, a different network size.